In [3]:
import re
import numpy as np
import pandas as pd

In [4]:
xls = pd.ExcelFile("TB_SP_RJ_casos_totais_municipio_ano_2001_2024.xlsx", engine="openpyxl")
xls.sheet_names

['casos_totais_long',
 'SP_indice_municipios',
 'SP_tabelas_casos_mes_ano',
 'SP_casos_totais_long',
 'SP_população',
 'RJ_indice_municipios',
 'RJ_tabelas_casos_mes_ano',
 'RJ_casos_totais_long',
 'RJ_população',
 'template_populacao',
 'base_modelagem_skeleton',
 'resumo_checks']

INTERPOLADOR LOG-LINEAR CAGR

In [5]:
INFILE = "TB_SP_RJ_casos_totais_municipio_ano_2001_2024.xlsx"
OUTFILE = "pop_interpolada_2000_2024_SP_RJ.xlsx"

SHEETS = ["SP_população", "RJ_população"]
YEARS = list(range(2000, 2025))

def _norm_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

def _find_col(df, patterns):
    for p in patterns:
        for c in df.columns:
            if re.search(p, c):
                return c
    return None

def _find_pop_col(df, year):
    # Aceita: "2000", "pop_2000", "pop2000", "população 2000", etc.
    yr = str(year)
    candidates = []
    for c in df.columns:
        if re.search(rf"\b{yr}\b", c) or re.search(rf"{yr}", c):
            candidates.append(c)
    # prioriza colunas com "pop" / "popul"
    for c in candidates:
        if re.search(r"pop|popul", c):
            return c
    return candidates[0] if candidates else None

def cagr_interp(p1, y1, p2, y2, y):
    """Interpolação/extrapolação log-linear (CAGR)."""
    if p1 is None or p2 is None or pd.isna(p1) or pd.isna(p2):
        return np.nan
    if p1 <= 0 or p2 <= 0 or y1 == y2:
        # fallback: linear (evita divisão por zero/negativos)
        return p1 + (p2 - p1) * ((y - y1) / (y2 - y1)) if y1 != y2 else p1
    r = (p2 / p1) ** (1.0 / (y2 - y1))
    return p1 * (r ** (y - y1))

def build_panel(df):
    df = _norm_cols(df)

    # tenta identificar colunas-chave
    code_col = _find_col(df, [r"cod.*ibge", r"codigo.*ibge", r"\bibge\b", r"cod.*mun", r"c[oó]d"])
    name_col = _find_col(df, [r"munic", r"muni", r"nome", r"municipio", r"município"])

    pop2000_col = _find_pop_col(df, 2000)
    pop2010_col = _find_pop_col(df, 2010)
    pop2022_col = _find_pop_col(df, 2022)

    for col in [pop2000_col, pop2010_col, pop2022_col]:
        if col is not None:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # monta saída wide
    out_rows = []
    for _, row in df.iterrows():
        p2000 = row[pop2000_col] if pop2000_col else np.nan
        p2010 = row[pop2010_col] if pop2010_col else np.nan
        p2022 = row[pop2022_col] if pop2022_col else np.nan

        # Estratégia por trechos:
        # 2000–2010: usa 2000->2010 se disponível, senão backcast a partir de 2010->2022
        # 2010–2022: usa 2010->2022 se disponível, senão 2000->2010
        # 2023–2024: extrapola usando 2010->2022 (preferencial) ou 2000->2010

        vals = {}
        for y in YEARS:
            if y <= 2010:
                if not pd.isna(p2000) and not pd.isna(p2010):
                    v = cagr_interp(p2000, 2000, p2010, 2010, y)
                elif not pd.isna(p2010) and not pd.isna(p2022):
                    # backcast usando 2010->2022
                    v = cagr_interp(p2010, 2010, p2022, 2022, y)
                else:
                    v = p2000 if not pd.isna(p2000) else (p2010 if not pd.isna(p2010) else p2022)
            elif y <= 2022:
                if not pd.isna(p2010) and not pd.isna(p2022):
                    v = cagr_interp(p2010, 2010, p2022, 2022, y)
                elif not pd.isna(p2000) and not pd.isna(p2010):
                    v = cagr_interp(p2000, 2000, p2010, 2010, y)
                else:
                    v = p2010 if not pd.isna(p2010) else (p2022 if not pd.isna(p2022) else p2000)
            else:  # 2023–2024
                if not pd.isna(p2010) and not pd.isna(p2022):
                    v = cagr_interp(p2010, 2010, p2022, 2022, y)
                elif not pd.isna(p2000) and not pd.isna(p2010):
                    v = cagr_interp(p2000, 2000, p2010, 2010, y)
                else:
                    v = p2022 if not pd.isna(p2022) else (p2010 if not pd.isna(p2010) else p2000)

            # arredonda e garante não-negativo
            if pd.isna(v):
                vals[y] = np.nan
            else:
                vals[y] = int(max(0, round(float(v))))

        base = {}
        if code_col: base[code_col] = row[code_col]
        if name_col: base[name_col] = row[name_col]
        base.update({str(y): vals[y] for y in YEARS})
        out_rows.append(base)

    wide = pd.DataFrame(out_rows)

    # monta long (município-ano-pop)
    id_cols = [c for c in [code_col, name_col] if c]
    long = wide.melt(id_vars=id_cols, var_name="ano", value_name="populacao")
    long["ano"] = pd.to_numeric(long["ano"], errors="coerce").astype("Int64")

    return wide, long

def main():
    with pd.ExcelFile(INFILE, engine="openpyxl") as xls:
        with pd.ExcelWriter(OUTFILE, engine="openpyxl") as writer:
            for sh in SHEETS:
                df = pd.read_excel(xls, sheet_name=sh)
                wide, long = build_panel(df)
                wide.to_excel(writer, sheet_name=f"{sh}_wide_2000_2024", index=False)
                long.to_excel(writer, sheet_name=f"{sh}_long_2000_2024", index=False)

    print(f"OK: arquivo gerado -> {OUTFILE}")

if __name__ == "__main__":
    main()

OK: arquivo gerado -> pop_interpolada_2000_2024_SP_RJ.xlsx
